# Proposal trait hero dengan NanoJev (Google Colab / Kaggle)

Notebook ini menjalankan **inference GPU** untuk membuat proposal trait berstatus `pending`. Ini bukan fine-tuning NanoJev. Tidak ada API inference, auto-approval, atau publikasi dataset. Checkpoint publik diunduh dari Hugging Face dan diverifikasi checksum-nya sebelum digunakan.

## Sebelum mulai

- Google Colab: pilih **Runtime → Change runtime type → GPU**.
- Kaggle: pilih **Settings → Accelerator → GPU** dan aktifkan Internet.
- Buat source bundle privat di laptop: `uv run --no-sync python notebooks/prepare_nanojev_source_bundle.py`. ZIP dibuat di `data/processed/nanojev_source_bundle.zip` dan hanya berisi pipeline serta mapping yang dibutuhkan.
- Colab akan meminta Anda mengunggah ZIP itu. Di Kaggle, unggah ZIP sebagai Dataset **Private**, tambahkan sebagai input notebook, lalu isi `SOURCE_BUNDLE_PATH` dengan path file di `/kaggle/input/...`. Notebook tidak clone repo atau mengakses GitHub.
- Jika sel batch terhenti tetapi runtime masih aktif, jalankan lagi pada direktori output yang sama; proposal valid yang sudah ada akan dilewati. Unduh ZIP hasil sebelum runtime notebook berakhir.

Setelah selesai, ekstrak file proposal dari ZIP ke `data/mappings/tactical_proposals/` di checkout lokal. Jalankan proses review lokal; jangan ubah status proposal menjadi approved sebelum memeriksa sumber dan labelnya.

In [ ]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import zipfile

SOURCE_BUNDLE_PATH = None  # Kaggle: set to /kaggle/input/<private-dataset-slug>/nanojev_source_bundle.zip.
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    SESSION_DIR = pathlib.Path("/kaggle/working")
    IS_COLAB = False
elif pathlib.Path("/content").exists():
    SESSION_DIR = pathlib.Path("/content")
    IS_COLAB = True
else:
    raise RuntimeError("Jalankan notebook ini di Google Colab atau Kaggle dengan GPU aktif.")

if SOURCE_BUNDLE_PATH:
    bundle_path = pathlib.Path(SOURCE_BUNDLE_PATH)
elif IS_COLAB:
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise RuntimeError("Unggah tepat satu file nanojev_source_bundle.zip.")
    bundle_path = pathlib.Path(zip_names[0])
else:
    raise RuntimeError("Atur SOURCE_BUNDLE_PATH ke ZIP yang dilampirkan sebagai private Kaggle Dataset.")
if not bundle_path.is_file():
    raise FileNotFoundError(f"Source bundle tidak ditemukan: {bundle_path}")

EXPECTED_FILES = {
    "packages/ml-engine/tactical_profiles.py",
    "packages/ml-engine/nanojev_provider.py",
    "packages/ml-engine/dataset_sources.py",
    "data/mappings/master_heroes_global.json",
    "data/mappings/hero_release_status.json",
    "data/mappings/hokstats_heroes.json",
    "data/mappings/hokstats_matchup_reviews.json",
    "data/mappings/hokstats_guide_reviews.json",
}
MANIFEST_FILE = "notebook_bundle_manifest.json"
REPO_DIR = SESSION_DIR / "hok-draft-engine-source"
REPO_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle_path) as archive:
    names = set(archive.namelist())
    if MANIFEST_FILE not in names:
        raise ValueError("ZIP tidak memiliki notebook_bundle_manifest.json; buat ulang source bundle.")
    bundle_manifest = json.loads(archive.read(MANIFEST_FILE))
    file_hashes = bundle_manifest.get("files", {})
    if bundle_manifest.get("schema_version") != 1 or set(file_hashes) != EXPECTED_FILES:
        raise ValueError("Daftar file source bundle tidak cocok dengan versi notebook ini.")
    if names != EXPECTED_FILES | {MANIFEST_FILE}:
        raise ValueError("ZIP berisi file tambahan atau file wajib hilang; buat ulang source bundle.")
    digest = hashlib.sha256()
    for relative, expected_hash in sorted(file_hashes.items()):
        content = archive.read(relative)
        actual_hash = hashlib.sha256(content).hexdigest()
        if actual_hash != expected_hash:
            raise ValueError(f"Checksum source tidak cocok: {relative}")
        digest.update(f"{relative}\0{actual_hash}\n".encode("utf-8"))
        destination = REPO_DIR / pathlib.Path(relative)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
    SOURCE_BUNDLE_ID = digest.hexdigest()
if SOURCE_BUNDLE_ID != bundle_manifest.get("bundle_id"):
    raise ValueError("ID source bundle tidak cocok dengan manifest.")

ML_DIR = REPO_DIR / "packages" / "ml-engine"
PROPOSAL_DIR = REPO_DIR / "data" / "mappings" / "tactical_proposals"
CHECKPOINT_DIR = REPO_DIR / "data" / "processed" / "nanojev"
print("Source bundle terverifikasi:", SOURCE_BUNDLE_ID)
print("File sumber:", len(EXPECTED_FILES))


In [ ]:
import torch
from packaging.version import Version

if not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA tidak terlihat. Aktifkan GPU accelerator pada runtime lalu restart sesi.")
if torch.version.cuda is None:
    raise RuntimeError("PyTorch pada runtime ini bukan build CUDA. Pilih runtime GPU dengan PyTorch CUDA.")
if Version(torch.__version__.split("+")[0]) < Version("2.15.0"):
    raise RuntimeError(f"Pipeline memerlukan PyTorch >=2.15.0 dengan CUDA; runtime menyediakan {torch.__version__}. Pilih runtime yang lebih baru.")

GPU_NAME = torch.cuda.get_device_name(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda}")
print(f"GPU: {GPU_NAME} | VRAM kosong {free_bytes / 1024**3:.1f} / {total_bytes / 1024**3:.1f} GiB")
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp32"
print("Precision inference:", PRECISION)

# Pertahankan build PyTorch CUDA yang disediakan runtime; pasang hanya dependensi model.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "transformers==5.17.0", "safetensors==0.8.0", "huggingface-hub"], check=True)
print("Dependensi NanoJev siap.")

## Unduh checkpoint terverifikasi

Sel berikut mengunduh bobot rilis NanoJev yang dipatok di source repo. Bobot berukuran sekitar 2,4 GB; jangan masukkan checkpoint ke ZIP proposal atau Git.

In [ ]:
download_cmd = [
    sys.executable, str(ML_DIR / "tactical_profiles.py"), "download",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
subprocess.run(download_cmd, cwd=ML_DIR, check=True)
print("Checkpoint siap; checksum rilis sudah diverifikasi.")

## Jalankan batch hero Global

Batch hanya memilih hero yang berstatus rilis Global dan memiliki bukti sumber. Proposal disimpan per hero dengan hash sumber. Profil yang belum punya bukti atau yang gagal akan tercatat di `coverage_report.json`.

In [ ]:
PROPOSAL_DIR.mkdir(parents=True, exist_ok=True)
report_path = PROPOSAL_DIR / "coverage_report.json"
batch_cmd = [
    sys.executable, str(ML_DIR / "tactical_profiles.py"), "propose-batch",
    "--output-dir", str(PROPOSAL_DIR),
    "--report", str(report_path),
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--precision", PRECISION,
]
print("Menjalankan:", " ".join(batch_cmd))
batch_result = subprocess.run(batch_cmd, cwd=ML_DIR)
print("Kode keluar batch:", batch_result.returncode)
if report_path.exists():
    report = __import__("json").loads(report_path.read_text(encoding="utf-8"))
    print("Ringkasan:", report.get("summary", {}))
if batch_result.returncode != 0:
    print("Batch tidak lengkap. Proposal parsial dan coverage_report tetap tersedia; perbaiki penyebabnya lalu jalankan ulang sel ini.")
else:
    print("Batch selesai. Semua file yang dibuat masih berstatus pending.")

In [ ]:
import json
import zipfile

run_info = {
    "source_bundle_id": SOURCE_BUNDLE_ID,
    "gpu": GPU_NAME,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "precision": PRECISION,
    "batch_exit_code": batch_result.returncode,
}
run_info_path = PROPOSAL_DIR / "run_info.json"
run_info_path.write_text(json.dumps(run_info, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
archive_path = SESSION_DIR / "nanojev_tactical_proposals.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(PROPOSAL_DIR.glob("*.json")):
        archive.write(path, arcname=path.name)
print(f"ZIP proposal tersimpan: {archive_path} ({archive_path.stat().st_size / 1024**2:.1f} MiB)")
if IS_COLAB:
    from google.colab import files
    files.download(str(archive_path))
else:
    print("Kaggle: unduh nanojev_tactical_proposals.zip dari panel Output / Files pada /kaggle/working.")